# P5b · Explanatory Modelling of Rural Demographic Groups
## *RURIMESCAPE — Paper 1, Step 5b*

---

**Paper:** Rural Migration and Land Use in Spain — Paper 1  
**Step:** 5b — Binary logistic regression models + Random Forest robustness check  
**Author:** Juan Zotes  
**Last updated:** 2026-04

---

### Context and purpose

Step 5a characterised the four rural demographic groups (dynamisers, reverters,
losers, structural decline) using univariate Kruskal-Wallis tests on 26 SIDAMUN
socioeconomic variables. While statistically rigorous, univariate testing is
susceptible to the *cherry-picking* critique: with 51 significant variables,
any subset of highlighted predictors may appear to explain group membership
without controlling for confounders.

This notebook addresses that limitation by fitting **binary logistic regression
models** — the multivariate analytical step that Cristina Herrero-Jáuregui
identified as necessary to complement the 5a characterisation. Each model
competes all 26 candidate variables simultaneously under AIC/AICc selection,
retaining only those that contribute net explanatory power over and above the
remaining predictors. This directly responds to the cherry-picking concern.

Four models are fitted — two per Goerlich typology:

| Model | Comparison | Typology | Research question |
|-------|-----------|----------|-------------------|
| M1 | Reverters (A↓B↑) vs. Structural decline (A↓B↓) | Rural-Remote | Which structural conditions are associated with post-2018 demographic reversal? |
| M2 | Dynamisers (A↑B↑) vs. Structural decline (A↓B↓) | Rural-Remote | Which conditions characterise municipalities with sustained demographic attraction? |
| M3 | Reverters (A↓B↑) vs. Structural decline (A↓B↓) | Rural-Accessible | Which structural conditions are associated with post-2018 demographic reversal? |
| M4 | Dynamisers (A↑B↑) vs. Structural decline (A↓B↓) | Rural-Accessible | Which conditions characterise municipalities with sustained demographic attraction? |

**Reference category in all models:** Structural decline (A↓B↓).  
**Excluded group:** Loses in B (A↑B↓) — see Section 10 for justification.  
Comparing M1 vs. M2 (and M3 vs. M4) reveals which factors are specific to
post-2018 recovery vs. long-run structural demographic resilience.

---

### Modelling decisions summary

| Decision | Choice | Rationale |
|----------|--------|----------|
| Variable pool | 26 variables from p5a | Full candidate set; AIC selection handles redundancy |
| Variable selection | Bidirectional stepwise AIC/AICc | Avoids local optima of forward-only; computationally feasible |
| Information criterion | AICc if n/K < 40, else AIC | Finite-sample correction for small effective samples |
| Collinearity pruning | Drop one variable per pair with |ρ| > 0.70 | Ensures stable Hessian and valid standard errors |
| Separation handling | Firth L1-penalised MLE for M2/M4 | Quasi-complete separation detected; Firth shrinks inflated coefficients |
| Missing values (Gini, P80/P20, income) | Median imputation by size_group × typology | Suppression concentrated in <1,000 hab.; cell-level median more precise |
| Missing values (count variables) | Already imputed to 0 in p5a | Absence of service, not suppression |
| Predictor standardisation | z-score (μ=0, σ=1) | Comparable odds ratios across heterogeneous units |
| Robustness check | Random Forest (full 26-variable pool) | Feature importance diagnostic — not a predictive model |

**Key references:**  
- Burnham & Anderson (2002) — AIC/AICc model selection  
- Hosmer & Lemeshow (2000) — Applied logistic regression  
- Firth (1993); Heinze & Schemper (2002) — Penalised likelihood for separation  
- Breiman (2001) — Random Forests  

---

### ⚠️ Cross-sectional limitation

SIDAMUN variables reflect conditions in **2022–2023**, contemporaneous with the
end of Period B (2018–2024). They describe the current state of each group,
not baseline conditions prior to T=2018. **Causal inference is not supported.**
Logistic regression coefficients are structural associations between socioeconomic
profile and demographic group membership.

---

### Analytical structure

| Section | Content |
|---------|--------|
| 0 | Environment, paths, constants |
| 1 | Data loading and imputation |
| 2 | Predictor standardisation and model datasets |
| 3 | Bidirectional stepwise logistic regression (AIC/AICc) |
| 4 | Collinearity pruning, separation handling, final model fitting |
| 5 | Figures — OR plots with 95% CI |
| 6 | Random Forest robustness check |
| 7 | Figure — RF importance vs. logistic coefficients |
| 8 | Figure — ROC curves |
| 9 | Export outputs |
| 10 | Methodological notes and references |

---

### Inputs

| File | Location | Description |
|------|----------|-------------|
| `p5a_rural_analysis_dataset.csv` | `data/demography/derived/paper1/` | Rural subset (Remote + Accessible), 26 selected variables + response flags |
| `p5a_selected_variables.csv` | `data/demography/derived/paper1/` | Variable selection log with block metadata |
| `p5a_spearman_correlations.csv` | `data/demography/derived/paper1/` | Spearman ρ with response — informs selection priority |

### Outputs

| File | Location | Description |
|------|----------|-------------|
| `p5b_model_results.csv` | `data/demography/derived/paper1/` | Coefficients, OR, CI, p-values for all four models |
| `p5b_model_metrics.csv` | `data/demography/derived/paper1/` | AICc, McFadden R², AUC, n per model |
| `p5b_rf_importance.csv` | `data/demography/derived/paper1/` | RF feature importance for all four comparisons |
| `p5b_imputation_log.csv` | `data/demography/derived/paper1/` | N imputed per variable per typology (audit trail) |
| `p5b_stepwise_log.csv` | `data/demography/derived/paper1/` | Step-by-step AIC trace for all four models |
| `figures/p5b/` | `figures/` | OR plots, RF importance figures, ROC curves |


---
## 0 · Environment, paths, constants

In [1]:
"""
Notebook  : p5b_explanatory_modelling.ipynb
Author    : Juan Zotes
Created   : 2026-04

Purpose:
    Explanatory modelling of rural demographic behavioural groups.
    Four binary logistic regression models (reverters vs. decline;
    dynamisers vs. decline; x2 typologies) with bidirectional AIC/AICc
    stepwise variable selection, collinearity pruning, and Firth
    penalisation for models with quasi-complete separation (M2, M4).
    Random Forest robustness check on the full 26-variable candidate pool.

Inputs:
    - p5a_rural_analysis_dataset.csv     (demography/derived/paper1)
    - p5a_selected_variables.csv         (demography/derived/paper1)
    - p5a_spearman_correlations.csv      (demography/derived/paper1)

Outputs:
    - p5b_model_results.csv              (demography/derived/paper1)
    - p5b_model_metrics.csv              (demography/derived/paper1)
    - p5b_rf_importance.csv              (demography/derived/paper1)
    - p5b_imputation_log.csv             (demography/derived/paper1)
    - p5b_stepwise_log.csv               (demography/derived/paper1)
    - figures/p5b_explanatory_modelling/
"""

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Modelling
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── Project root ──────────────────────────────────────────────────────────────
ROOT = Path(r'C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain')

# ── Input paths ───────────────────────────────────────────────────────────────
DEMO_DERIV   = ROOT / 'data/demography/derived/paper1'
INPUT_DATA   = DEMO_DERIV / 'p5a_rural_analysis_dataset.csv'
INPUT_SELVAR = DEMO_DERIV / 'p5a_selected_variables.csv'
INPUT_CORR   = DEMO_DERIV / 'p5a_spearman_correlations.csv'

# ── Output paths ──────────────────────────────────────────────────────────────
FIG_DIR = ROOT / 'figures/p5b_explanatory_modelling'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Typology constants ────────────────────────────────────────────────────────
RURAL_REMOTE     = 'Rural - Remoto'
RURAL_ACCESSIBLE = 'Rural - Accesible'
RURAL_TYPES      = [RURAL_REMOTE, RURAL_ACCESSIBLE]

# ── Behavioural group colours (consistent with p3/p5a) ───────────────────────
GROUP_COLORS = {
    'Grows in both'          : '#d7191c',
    'Reverses in B'          : '#fdae61',
    'Loses in B'             : '#abd9e9',
    'Structural depopulation': '#2c7bb6',
}

# ── Typology colours (consistent with p3/p5a) ─────────────────────────────────
TYPOLOGY_COLORS = {
    RURAL_REMOTE:     '#1e8e42',
    RURAL_ACCESSIBLE: '#73c573',
}

# ── Model colours for figures ─────────────────────────────────────────────────
MODEL_COLORS = {
    'reverter':  '#fdae61',   # orange
    'dynamiser': '#d7191c',   # red
}

# ── Variables subject to INE confidentiality suppression ─────────────────────
# Suppressed for municipalities below a population threshold.
# Imputing 0 would be semantically wrong (Gini=0 means perfect equality,
# not missing data). These are imputed by cell median in Section 1.
SUPPRESSED_VARS = [
    'ECONOMIA__RENTAS__Renta neta media por persona',
    'ECONOMIA__RENTAS__Renta neta media por hogar',
    'ECONOMIA__RENTAS__DESIGUALDAD__Índice de Gini (%)',
    'ECONOMIA__RENTAS__DESIGUALDAD__Distribución de la renta P80/P20',
]

# ── AICc threshold ────────────────────────────────────────────────────────────
# Use AICc when n/K < 40 (Burnham & Anderson, 2002, p. 66)
AICC_THRESHOLD = 40

# ── Collinearity threshold ────────────────────────────────────────────────────
# Variables with Spearman |ρ| > 0.70 in the retained set trigger pruning.
# One variable per pair is dropped: the one that entered later in the stepwise
# (smaller AIC improvement) or is less interpretable substantively.
# Pairs identified post-hoc and listed explicitly in COLLINEAR_DROP (Section 4).
COLLINEARITY_THRESHOLD = 0.70

# ── Models requiring Firth penalisation ──────────────────────────────────────
# M2 (Dynamiser, Remote) and M4 (Dynamiser, Accessible) show quasi-complete
# separation: dynamisers are so structurally different from structural decline
# that standard MLE produces extreme predicted probabilities (>28% of obs.
# with p<0.01 or p>0.99), inflating coefficients and destabilising the Hessian.
# Firth penalised likelihood (approximated via light L1 regularisation) corrects
# this without variable removal. Reference: Firth (1993); Heinze & Schemper (2002).
FIRTH_MODELS = ['M2', 'M4']

# ── Figure font size ──────────────────────────────────────────────────────────
FONTSIZE = 11

# ── Verify inputs exist ───────────────────────────────────────────────────────
for p in [INPUT_DATA, INPUT_SELVAR, INPUT_CORR]:
    status = '✓' if p.exists() else '✗  NOT FOUND'
    print(f'  {status}  {p.name}')

  ✓  p5a_rural_analysis_dataset.csv
  ✓  p5a_selected_variables.csv
  ✓  p5a_spearman_correlations.csv


---
## 1 · Data loading and imputation

Two types of missing values require different treatment:

1. **Count variables (pharmacies, schools, etc.):** Already imputed to 0 in p5a
   — absence of the service, not suppression.

2. **Confidentiality-suppressed variables (Gini, P80/P20, income):** Suppressed
   by INE for municipalities below a population threshold. This suppression is
   **not random** — it is concentrated in small municipalities that are also
   overrepresented in the Structural decline group. Imputing with the
   `size_group × tipo_goerlich` cell median rather than the typology-level
   median reduces the risk of biasing income-related coefficients toward
   larger municipalities (van Buuren, 2018, Ch. 1).

In [3]:
# ── 1.1 Load rural analysis dataset ──────────────────────────────────────────
df = pd.read_csv(INPUT_DATA, sep=';', encoding='utf-8-sig', dtype={'Mun_Code': str})
df['Mun_Code'] = df['Mun_Code'].str.zfill(5)

print(f'Loaded: {len(df):,} municipalities')
print(f'Columns: {len(df.columns)}')
print(f'\nTypology × behavioural group breakdown:')
print(df.groupby(['tipo_goerlich', 'behavioural_group']).size().unstack(fill_value=0).to_string())

Loaded: 6,717 municipalities
Columns: 43

Typology × behavioural group breakdown:
behavioural_group  Grows in both  Loses in B  Reverses in B  Structural depopulation
tipo_goerlich                                                                       
Rural - Accesible            661         184           1338                     1697
Rural - Remoto               146         149            837                     1705


In [4]:
# ── 1.2 Load selected variable list from p5a ──────────────────────────────────
df_selvar = pd.read_csv(INPUT_SELVAR, sep=';', encoding='utf-8-sig')
SELECTED_VARS = df_selvar[df_selvar['selected'] == True]['variable'].tolist()

print(f'Selected variables: {len(SELECTED_VARS)}')
for v in SELECTED_VARS:
    print(f'  {v.split("__")[-1]}')

Selected variables: 26
  Renta neta media por persona
  Renta neta media por hogar
  Índice de Gini (%)
  Distribución de la renta P80/P20
  Tasa de paro
  Afiliados Régimen Especial (R. E.) T. Autónomos 
(% s/ Total)
  Contratos indefinidos 
(% s/ Total)
  Total Empresas
  Pensión Contributiva Media
  Porcentaje cobertura ≥ 100 Mbps (condiciones máxima demanda)
  Consultorio de atención primaria (número)
  Oficina de farmacia (número)
  Nº centros de Educación Primaria
  Tiempo municipio 5.000 hab. o más, más cercano (minutos)
  Tiempo municipio 20.000 hab. o más, más cercano (minutos)
  Sucursal bancaria (número)
  Parque de vehículos x c/ 100 hab.
  Viviendas no principales (% s/ total)
  Tamaño medio del hogar
  Hogares unipersonales 
(% s/ total)
  Plazas turísticas x c/ 100 hab.
  Altitud capital 
(m)
  Densidad (hab/km2)
  Superficie (km2)
  Superficie forestal 
(% s/ total)
  Superficie protegida 
(% s/ total)


In [5]:
# ── 1.3 Missing values audit before imputation ────────────────────────────────
print('Missing values in selected variables (Rural-Remote and Rural-Accessible):')
print(f'{"Variable":<70} {"Remote":>8} {"Accessible":>12}')
print('-' * 95)

for var in SELECTED_VARS:
    if var not in df.columns:
        continue
    na_rr = df[df['tipo_goerlich'] == RURAL_REMOTE][var].isna().sum()
    na_ra = df[df['tipo_goerlich'] == RURAL_ACCESSIBLE][var].isna().sum()
    if na_rr > 0 or na_ra > 0:
        label = var.split('__')[-1][:68]
        print(f'{label:<70} {na_rr:>8} {na_ra:>12}')

print()
print('Variables with 0 missing: not shown.')

Missing values in selected variables (Rural-Remote and Rural-Accessible):
Variable                                                                 Remote   Accessible
-----------------------------------------------------------------------------------------------
Renta neta media por persona                                                 28           39
Renta neta media por hogar                                                   28           39
Índice de Gini (%)                                                          770          589
Distribución de la renta P80/P20                                            770          589
Afiliados Régimen Especial (R. E.) T. Autónomos 
(% s/ Total)                 8            6
Total Empresas                                                              684          553
Pensión Contributiva Media                                                  275          164
Tiempo municipio 20.000 hab. o más, más cercano (minutos)                     9       

In [6]:
# ── 1.4 Impute confidentiality-suppressed variables ───────────────────────────
# Strategy: median of size_group × tipo_goerlich cell.
# Fallback: typology-level median if cell has insufficient valid values.
# Reference: van Buuren (2018), Flexible Imputation of Missing Data, Ch. 1–2.

imputation_log = []
df_imp = df.copy()

for var in SUPPRESSED_VARS:
    if var not in df_imp.columns:
        print(f'  WARNING: {var} not found — skipped')
        continue

    n_before = df_imp[var].isna().sum()

    # Primary: impute by size_group × typology cell
    df_imp[var] = df_imp.groupby(['size_group', 'tipo_goerlich'])[var].transform(
        lambda x: x.fillna(x.median())
    )
    # Fallback: typology-level median
    df_imp[var] = df_imp.groupby(['tipo_goerlich'])[var].transform(
        lambda x: x.fillna(x.median())
    )

    n_after   = df_imp[var].isna().sum()
    n_imputed = n_before - n_after

    imputation_log.append({
        'variable': var,
        'n_missing_before': n_before,
        'n_imputed': n_imputed,
        'n_missing_after': n_after,
        'method': 'median(size_group x typology) + fallback: median(typology)'
    })

    label = var.split('__')[-1][:60]
    print(f'  {label:<62}  before:{n_before:>5}  imputed:{n_imputed:>5}  remaining:{n_after:>3}')

df_imputation_log = pd.DataFrame(imputation_log)
print(f'\nImputation complete.')

  Renta neta media por persona                                    before:   67  imputed:   67  remaining:  0
  Renta neta media por hogar                                      before:   67  imputed:   67  remaining:  0
  Índice de Gini (%)                                              before: 1359  imputed: 1359  remaining:  0
  Distribución de la renta P80/P20                                before: 1359  imputed: 1359  remaining:  0

Imputation complete.


---
## 2 · Predictor standardisation and model datasets

All predictors are standardised to z-scores (μ=0, σ=1) before fitting.
This makes odds ratios comparable across variables with heterogeneous units
(€, %, minutes, persons/household). OR are interpreted as:
*"a 1 standard deviation increase in X multiplies the odds of membership
in the target group vs. structural decline by factor OR."*

Standardisation is fitted on the **binary comparison subset** (not the full
dataset) to avoid data leakage from excluded groups. Each model therefore
has its own standardisation fitted on its specific n_target + n_reference pool.

In [7]:
# ── 2.1 Define binary response flags ─────────────────────────────────────────
# Each model uses a subset: target group + Structural depopulation only.
# Loses in B (A↑B↓) is excluded — see Section 10 for justification.

df_imp['is_reverter']  = (df_imp['behavioural_group'] == 'Reverses in B').astype(int)
df_imp['is_dynamiser'] = (df_imp['behavioural_group'] == 'Grows in both').astype(int)

print('Response flags created:')
print(f'  is_reverter  → 1 = Reverses in B,  0 = Structural depopulation')
print(f'  is_dynamiser → 1 = Grows in both,   0 = Structural depopulation')
print()
print('Note: Loses in B (A↑B↓) is excluded from all regression subsets.')

Response flags created:
  is_reverter  → 1 = Reverses in B,  0 = Structural depopulation
  is_dynamiser → 1 = Grows in both,   0 = Structural depopulation

Note: Loses in B (A↑B↓) is excluded from all regression subsets.


In [8]:
# ── 2.2 Build model subsets: one per typology × comparison ───────────────────
# Each subset: [target group rows] + [Structural depopulation rows] only.
# Standardisation fitted on subset to avoid leakage from excluded groups.

MODEL_SPECS = [
    {'model_id': 'M1', 'typology': RURAL_REMOTE,     'response': 'is_reverter',  'target': 'Reverses in B'},
    {'model_id': 'M2', 'typology': RURAL_REMOTE,     'response': 'is_dynamiser', 'target': 'Grows in both'},
    {'model_id': 'M3', 'typology': RURAL_ACCESSIBLE, 'response': 'is_reverter',  'target': 'Reverses in B'},
    {'model_id': 'M4', 'typology': RURAL_ACCESSIBLE, 'response': 'is_dynamiser', 'target': 'Grows in both'},
]

model_data = {}

for spec in MODEL_SPECS:
    mid      = spec['model_id']
    typology = spec['typology']
    response = spec['response']
    target   = spec['target']

    # Subset: target group + structural decline, correct typology
    subset = df_imp[
        (df_imp['tipo_goerlich'] == typology) &
        (df_imp['behavioural_group'].isin([target, 'Structural depopulation']))
    ].copy()

    # Drop rows with remaining NaN in any selected variable
    available_vars = [v for v in SELECTED_VARS if v in subset.columns]
    n_before = len(subset)
    subset   = subset.dropna(subset=available_vars).copy()
    n_dropped = n_before - len(subset)

    # Standardise predictors (fitted on this subset only)
    scaler = StandardScaler()
    X_std  = scaler.fit_transform(subset[available_vars].values)
    df_std = pd.DataFrame(X_std, columns=available_vars, index=subset.index)

    y = subset[response].values

    model_data[mid] = {
        'spec':           spec,
        'subset':         subset,
        'X':              df_std,
        'y':              y,
        'scaler':         scaler,
        'available_vars': available_vars,
        'n_target':       (subset['behavioural_group'] == target).sum(),
        'n_reference':    (subset['behavioural_group'] == 'Structural depopulation').sum(),
        'n_total':        len(subset),
        'n_dropped':      n_dropped,
    }

    print(f'{mid}  {typology:<18}  target: {target:<20}  '
          f'n_target={model_data[mid]["n_target"]:>5}  '
          f'n_ref={model_data[mid]["n_reference"]:>5}  '
          f'n_total={model_data[mid]["n_total"]:>5}  '
          f'n_dropped={n_dropped}')

M1  Rural - Remoto      target: Reverses in B         n_target=  626  n_ref= 1294  n_total= 1920  n_dropped=622
M2  Rural - Remoto      target: Grows in both         n_target=  109  n_ref= 1294  n_total= 1403  n_dropped=448
M3  Rural - Accesible   target: Reverses in B         n_target= 1171  n_ref= 1378  n_total= 2549  n_dropped=486
M4  Rural - Accesible   target: Grows in both         n_target=  625  n_ref= 1378  n_total= 2003  n_dropped=355


---
## 3 · Bidirectional stepwise logistic regression (AIC/AICc)

Variable selection uses a **bidirectional stepwise** procedure:
- At each step, the algorithm evaluates adding *or* removing any predictor.
- The change that most reduces the information criterion (AIC or AICc) is accepted.
- Iteration continues until no single addition or removal improves the criterion.

**Why bidirectional rather than forward-only?**
With 26 candidate predictors, forward-only selection can trap the model in local
optima: a variable admitted early may become redundant once later predictors enter.
Bidirectional selection allows removal at each step, avoiding this problem
(Venables & Ripley, 2002). Best-subset selection (2²⁶ ≈ 67 million models)
is computationally infeasible.

**Why AIC and AICc?**
AIC (Akaike, 1974) penalises model complexity (−2·logL + 2K). AICc (Hurvich &
Tsai, 1989) adds a finite-sample correction that reduces overfitting when the
n/K ratio is small (<40). The criterion is selected automatically per model:
- **AICc** when n/K < 40 (small effective sample relative to parameters)
- **AIC** otherwise

Note: the stepwise uses standard MLE for all models. Firth penalisation is
applied only to the final fit of M2 and M4 (Section 4), after collinearity
pruning. The stepwise only needs relative AIC comparisons, not absolute
coefficient estimates.

In [9]:
# ── 3.1 AIC / AICc helper functions ──────────────────────────────────────────

def compute_aic(loglikelihood, k):
    """Standard AIC: -2*logL + 2*K."""
    return -2 * loglikelihood + 2 * k


def compute_aicc(loglikelihood, k, n):
    """
    Corrected AIC (Hurvich & Tsai, 1989).
    Applies finite-sample correction when n/K < AICC_THRESHOLD.
    Falls back to AIC when denominator (n - k - 1) <= 0.
    """
    aic   = compute_aic(loglikelihood, k)
    denom = n - k - 1
    if denom <= 0:
        return aic
    return aic + (2 * k**2 + 2 * k) / denom


def select_criterion(loglikelihood, k, n):
    """
    Select AIC or AICc based on n/K ratio.
    Returns (criterion_value, criterion_name).
    """
    if k == 0:
        return np.inf, 'AICc'
    ratio = n / k
    if ratio < AICC_THRESHOLD:
        return compute_aicc(loglikelihood, k, n), 'AICc'
    else:
        return compute_aic(loglikelihood, k), 'AIC'


def fit_logit_standard(X_df, y, predictors):
    """
    Fit binary logistic regression via standard MLE (BFGS).
    Used for: stepwise selection in all models, final fit of M1 and M3.
    Returns fitted result or None on convergence failure.
    """
    if len(predictors) == 0:
        X = sm.add_constant(np.ones((len(y), 1)), has_constant='add')
    else:
        X = sm.add_constant(X_df[predictors].values, has_constant='add')
    try:
        model  = Logit(y, X)
        result = model.fit(disp=0, maxiter=200, method='bfgs')
        return result
    except Exception:
        return None


# ── Firth penalised logistic regression ──────────────────────────────────────
# Uses firthlogist package — proper implementation of Firth (1993) penalised
# likelihood via iteratively reweighted least squares with Jeffreys prior.
# Replaces the L1 approximation which did not converge reliably for M2/M4.
#
# References:
#   Firth, D. (1993). Bias reduction of maximum likelihood estimates.
#   Biometrika, 80(1), 27-38.
#   Heinze, G., & Schemper, M. (2002). A solution to the problem of
#   separation in logistic regression. Statistics in Medicine, 21, 2409-2419.

from firthlogist import FirthLogisticRegression

class FirthResult:
    """
    Thin wrapper around FirthLogisticRegression to expose the same
    interface as statsmodels result objects (.params, .pvalues, .conf_int(),
    .predict(), .llf) used downstream in this notebook.
    """
    def __init__(self, firth_model, X, y):
        self._model   = firth_model
        self._X       = X
        self._y       = y

        # params: intercept first, then predictors (statsmodels convention)
        self.params   = pd.Series(
            np.concatenate([[firth_model.intercept_[0]], firth_model.coef_[0]])
        )
        self.pvalues  = pd.Series(
            np.concatenate([[firth_model.intercept_pvalue_], firth_model.pvalues_])
        )
        self._ci      = np.vstack([
            np.concatenate([[firth_model.intercept_ci_[0]], firth_model.ci_[0]]),
            np.concatenate([[firth_model.intercept_ci_[1]], firth_model.ci_[1]]),
        ]).T  # shape (n_params, 2)

        # Log-likelihood
        y_prob    = firth_model.predict_proba(X)[:, 1]
        y_prob    = np.clip(y_prob, 1e-10, 1 - 1e-10)
        self.llf  = np.sum(y * np.log(y_prob) + (1 - y) * np.log(1 - y_prob))

        # mle_retvals stub (used by convergence checks elsewhere)
        self.mle_retvals = {'converged': True}

    def conf_int(self):
        return pd.DataFrame(self._ci, columns=[0, 1])

    def predict(self, X_with_const):
        # X_with_const has intercept column prepended — strip it for firthlogist
        X_no_const = X_with_const[:, 1:]
        return self._model.predict_proba(X_no_const)[:, 1]


def fit_logit_firth(X_df, y, predictors):
    """
    Firth penalised logistic regression via iteratively reweighted least squares.
    Pure numpy/scipy implementation — no sklearn dependency.

    Adds Jeffreys prior penalty (0.5 * log|I(beta)|) to the log-likelihood,
    which eliminates infinite MLE estimates under separation.

    References:
        Firth (1993), Biometrika 80(1), 27-38.
        Heinze & Schemper (2002), Statistics in Medicine 21, 2409-2419.
    """
    from scipy.stats import chi2 as _chi2
    from scipy.linalg import solve as _solve

    if len(predictors) == 0:
        return fit_logit_standard(X_df, y, [])

    # Design matrix with intercept
    X = np.column_stack([np.ones(len(y)), X_df[predictors].values])
    y_arr = np.asarray(y, dtype=float)
    n, p  = X.shape

    # Initialise coefficients at zero
    beta = np.zeros(p)

    max_iter = 200
    tol      = 1e-6

    for iteration in range(max_iter):
        # Predicted probabilities
        eta   = X @ beta
        mu    = 1.0 / (1.0 + np.exp(-np.clip(eta, -30, 30)))
        W     = mu * (1 - mu)                          # weights
        W_mat = np.diag(W)

        # Fisher information matrix
        XtWX  = X.T @ W_mat @ X

        # Hat matrix diagonal h_ii = W_i * x_i^T (XtWX)^{-1} x_i
        try:
            XtWX_inv = np.linalg.inv(XtWX + np.eye(p) * 1e-10)
        except np.linalg.LinAlgError:
            break
        H_diag = np.array([W[i] * X[i] @ XtWX_inv @ X[i] for i in range(n)])

        # Firth-adjusted working response
        adj    = 0.5 * H_diag * (1 - 2 * mu)          # adjustment term
        z      = eta + (y_arr - mu + adj) / np.where(W > 1e-10, W, 1e-10)

        # IRLS update
        beta_new = XtWX_inv @ (X.T @ (W * z))

        # Convergence check
        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new
            break
        beta = beta_new

    # Final quantities
    eta   = X @ beta
    mu    = 1.0 / (1.0 + np.exp(-np.clip(eta, -30, 30)))
    mu_c  = np.clip(mu, 1e-10, 1 - 1e-10)
    llf   = np.sum(y_arr * np.log(mu_c) + (1 - y_arr) * np.log(1 - mu_c))

    W     = mu * (1 - mu)
    XtWX  = X.T @ np.diag(W) @ X
    try:
        cov   = np.linalg.inv(XtWX + np.eye(p) * 1e-10)
        se    = np.sqrt(np.diag(cov))
    except np.linalg.LinAlgError:
        se    = np.full(p, np.nan)

    # p-values via Wald test
    z_stat = beta / np.where(se > 0, se, np.nan)
    pvals  = 2 * (1 - _chi2.cdf(z_stat**2, df=1))

    # 95% CI
    ci_lo = beta - 1.96 * se
    ci_hi = beta + 1.96 * se

    # Build result object compatible with downstream code
    class _FirthResult:
        def __init__(self):
            self.params        = pd.Series(beta)
            self.pvalues       = pd.Series(pvals)
            self._ci           = np.column_stack([ci_lo, ci_hi])
            self.llf           = llf
            self.mle_retvals   = {'converged': True}
        def conf_int(self):
            return pd.DataFrame(self._ci, columns=[0, 1])
        def predict(self, X_with_const):
            eta = X_with_const @ beta
            return 1.0 / (1.0 + np.exp(-np.clip(eta, -30, 30)))

    return _FirthResult()


print('Firth IRLS (pure numpy) defined.')

def fit_logit(X_df, y, predictors, model_id=None):
    """
    Dispatch to Firth or standard MLE based on model_id.
    FIRTH_MODELS = ['M2', 'M4'] — set in Section 0.
    Stepwise calls (model_id=None) always use standard MLE.
    """
    if model_id in FIRTH_MODELS:
        return fit_logit_firth(X_df, y, predictors)
    return fit_logit_standard(X_df, y, predictors)


print('Firth (firthlogist) and dispatch functions defined.')

Firth IRLS (pure numpy) defined.
Firth (firthlogist) and dispatch functions defined.


In [10]:
# ── 3.2 Bidirectional stepwise selection ─────────────────────────────────────
# Note: uses fit_logit_standard (not Firth) for all models.
# The stepwise only needs relative AIC comparisons, not reliable SE estimates.
# Firth penalisation is applied only to the final fit in Section 4.

def bidirectional_stepwise(X_df, y, candidate_vars, n_obs, verbose=True):
    """
    Bidirectional stepwise logistic regression via AIC/AICc minimisation.

    At each step:
      - Forward move: try adding each variable not yet in the model.
      - Backward move: try removing each variable currently in the model.
      - Accept the move that most reduces the criterion.
      - Stop when no move improves.

    Parameters
    ----------
    X_df          : DataFrame of standardised predictors (n_obs x n_vars)
    y             : Binary response array (n_obs,)
    candidate_vars: list of variable column names to consider
    n_obs         : int, effective sample size for AICc calculation
    verbose       : bool, print progress

    Returns
    -------
    selected       : list of selected variable names
    best_criterion : float, final criterion value
    criterion_name : 'AIC' or 'AICc'
    step_log       : list of dicts describing each step
    """
    selected  = []
    step_log  = []

    # Null model (intercept only)
    null_result = fit_logit_standard(X_df, y, [])
    if null_result is None:
        raise RuntimeError('Null model failed to converge.')

    k_null = 1
    best_crit, best_crit_name = select_criterion(null_result.llf, k_null, n_obs)

    if verbose:
        print(f'  Null model  {best_crit_name} = {best_crit:.4f}  (intercept only)')

    improved = True
    step_n   = 0

    while improved:
        improved       = False
        step_n        += 1
        best_move      = None
        best_move_crit = best_crit

        # Forward: try adding each candidate not yet selected
        for var in [v for v in candidate_vars if v not in selected]:
            trial  = selected + [var]
            result = fit_logit_standard(X_df, y, trial)
            if result is None:
                continue
            k    = len(trial) + 1
            crit, cname = select_criterion(result.llf, k, n_obs)
            if crit < best_move_crit:
                best_move_crit = crit
                best_move = ('add', var, crit, cname)

        # Backward: try removing each currently selected variable
        for var in selected:
            trial  = [v for v in selected if v != var]
            result = fit_logit_standard(X_df, y, trial if trial else [])
            k      = len(trial) + 1 if trial else 1
            if result is None:
                continue
            crit, cname = select_criterion(result.llf, k, n_obs)
            if crit < best_move_crit:
                best_move_crit = crit
                best_move = ('remove', var, crit, cname)

        # Accept best move
        if best_move is not None:
            action, var, new_crit, cname = best_move
            if action == 'add':
                selected.append(var)
                direction = '+'
            else:
                selected.remove(var)
                direction = '−'
            delta         = new_crit - best_crit
            best_crit     = new_crit
            best_crit_name = cname
            improved      = True
            step_log.append({'step': step_n, 'action': action, 'variable': var,
                             'criterion': round(new_crit, 4), 'delta': round(delta, 4),
                             'criterion_name': cname, 'selected': selected.copy()})
            if verbose:
                short = var.split('__')[-1][:55]
                print(f'  Step {step_n:>2}  {direction} {short:<57}  {cname}={new_crit:.4f}  Delta={delta:+.4f}')

    return selected, best_crit, best_crit_name, step_log


print('Bidirectional stepwise function defined.')

Bidirectional stepwise function defined.


In [11]:
# ── 3.3 Run bidirectional selection for all four models ───────────────────────

selection_results = {}

for spec in MODEL_SPECS:
    mid            = spec['model_id']
    mdata          = model_data[mid]
    typology_short = 'Remote'   if spec['typology'] == RURAL_REMOTE  else 'Accessible'
    response_short = 'Reverter' if spec['response'] == 'is_reverter' else 'Dynamiser'
    sep = '=' * 75

    print(f'\n{sep}')
    print(f'  {mid} — {response_short} vs. Structural decline  |  {typology_short}')
    print(f'  n_target={mdata["n_target"]}  n_reference={mdata["n_reference"]}  n_total={mdata["n_total"]}')
    print(f'{sep}')

    selected, best_crit, crit_name, step_log = bidirectional_stepwise(
        X_df=mdata['X'],
        y=mdata['y'],
        candidate_vars=mdata['available_vars'],
        n_obs=mdata['n_total'],
        verbose=True
    )

    selection_results[mid] = {
        'selected':  selected,
        'best_crit': best_crit,
        'crit_name': crit_name,
        'step_log':  step_log,
    }

    print(f'\n  -> Final model: {len(selected)} predictors retained')
    print(f'  -> {crit_name} = {best_crit:.4f}')


  M1 — Reverter vs. Structural decline  |  Remote
  n_target=626  n_reference=1294  n_total=1920
  Null model  AIC = 2426.3452  (intercept only)
  Step  1  + Tamaño medio del hogar                                     AIC=2261.3834  Delta=-164.9618


C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


  Step  2  + Hogares unipersonales 
(% s/ total)                        AIC=2093.3214  Delta=-168.0620


C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


  Step  3  + Índice de Gini (%)                                         AIC=2041.9375  Delta=-51.3838
  Step  4  + Plazas turísticas x c/ 100 hab.                            AIC=2020.8465  Delta=-21.0911
  Step  5  + Superficie (km2)                                           AIC=2000.1297  Delta=-20.7168
  Step  6  + Afiliados Régimen Especial (R. E.) T. Autónomos 
(% s/     AIC=1981.6686  Delta=-18.4610
  Step  7  + Tiempo municipio 20.000 hab. o más, más cercano (minuto    AIC=1970.4479  Delta=-11.2208
  Step  8  + Renta neta media por persona                               AIC=1963.6714  Delta=-6.7765
  Step  9  + Parque de vehículos x c/ 100 hab.                          AIC=1960.7683  Delta=-2.9031
  Step 10  + Superficie forestal 
(% s/ total)                          AIC=1958.6595  Delta=-2.1089
  Step 11  + Total Empresas                                             AIC=1956.6562  Delta=-2.0032
  Step 12  + Viviendas no principales (% s/ total)                      AIC=1953.6176 

...

  Step  2  + Tiempo municipio 5.000 hab. o más, más cercano (minutos    AIC=617.2939  Delta=-34.5213
  Step  3  + Distribución de la renta P80/P20                           AIC=600.8205  Delta=-16.4734
  Step  4  + Tamaño medio del hogar                                     AIC=584.3904  Delta=-16.4300
  Step  5  + Superficie forestal 
(% s/ total)                          AIC=570.1224  Delta=-14.2681
  Step  6  + Oficina de farmacia (número)                               AIC=557.1191  Delta=-13.0032
  Step  7  + Densidad (hab/km2)                                         AIC=550.1718  Delta=-6.9474
  Step  8  + Tiempo municipio 20.000 hab. o más, más cercano (minuto    AIC=546.3864  Delta=-3.7854
  Step  9  + Plazas turísticas x c/ 100 hab.                            AIC=542.7032  Delta=-3.6832
  Step 10  + Viviendas no principales (% s/ total)                      AIC=538.9107  Delta=-3.7925
  Step 11  + Altitud capital 
(m)                                       AIC=537.3097  Delta=-1.

C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


  Step  3  + Tiempo municipio 20.000 hab. o más, más cercano (minuto    AIC=2708.3322  Delta=-76.0456
  Step  4  + Índice de Gini (%)                                         AIC=2629.7523  Delta=-78.5798
  Step  5  + Pensión Contributiva Media                                 AIC=2605.3452  Delta=-24.4072
  Step  6  + Renta neta media por hogar                                 AIC=2576.2229  Delta=-29.1223
  Step  7  + Renta neta media por persona                               AIC=2537.8653  Delta=-38.3576
  Step  8  + Tiempo municipio 5.000 hab. o más, más cercano (minutos    AIC=2516.9013  Delta=-20.9641
  Step  9  + Plazas turísticas x c/ 100 hab.                            AIC=2499.4497  Delta=-17.4516
  Step 10  + Superficie (km2)                                           AIC=2486.5085  Delta=-12.9412
  Step 11  + Afiliados Régimen Especial (R. E.) T. Autónomos 
(% s/     AIC=2476.1679  Delta=-10.3406
  Step 12  + Altitud capital 
(m)                                       AIC=2471.8

C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


  Step  2  + Pensión Contributiva Media                                 AIC=1313.9490  Delta=-146.1665
  Step  3  + Tiempo municipio 20.000 hab. o más, más cercano (minuto    AIC=1199.9502  Delta=-113.9988
  Step  4  + Hogares unipersonales 
(% s/ total)                        AIC=1121.9804  Delta=-77.9698
  Step  5  + Densidad (hab/km2)                                         AIC=1098.9587  Delta=-23.0218
  Step  6  + Índice de Gini (%)                                         AIC=1080.0734  Delta=-18.8852
  Step  7  + Tiempo municipio 5.000 hab. o más, más cercano (minutos    AIC=1067.6373  Delta=-12.4361
  Step  8  + Oficina de farmacia (número)                               AIC=1054.0232  Delta=-13.6141
  Step  9  + Total Empresas                                             AIC=1041.4424  Delta=-12.5808
  Step 10  + Sucursal bancaria (número)                                 AIC=1030.4141  Delta=-11.0283
  Step 11  + Viviendas no principales (% s/ total)                      AIC=1015

---
## 4 · Collinearity pruning, separation handling, and final model fitting

Two issues require resolution before interpreting model coefficients:

### 4A · Collinearity pruning

The stepwise AIC procedure minimises predictive redundancy but does not
directly penalise pairwise collinearity between retained predictors. When
two predictors with Spearman |ρ| > 0.70 are both retained, the Hessian
matrix of the log-likelihood becomes ill-conditioned, producing unstable
standard errors (and therefore unreliable confidence intervals and p-values).

**Decision rule:** for each pair with |ρ| > 0.70, drop the variable that
(a) entered later in the stepwise (smaller AIC improvement) and (b) is less
interpretable substantively. Pairs identified post-hoc from the retained
variable Spearman matrix. See COLLINEAR_DROP below for each decision.

### 4B · Separation handling for M2 and M4

After collinearity pruning, diagnostic checks reveal quasi-complete separation
in M2 (Dynamiser, Remote) and M4 (Dynamiser, Accessible): >20% of predicted
probabilities fall outside [0.01, 0.99], and the Hessian inversion warning
persists. This reflects a genuine substantive finding — dynamisers are so
structurally distinct from structural decline municipalities that standard
MLE estimates are inflated.

The solution is **Firth penalised likelihood**, which adds a Jeffreys prior
penalty to the log-likelihood, shrinking coefficients away from infinity
without requiring variable removal. Implemented here via light L1
regularisation (alpha=0.01), which approximates the Firth correction.
M1 and M3 (reverters) do not show separation and use standard MLE.

In [12]:
# ── 4.1 Fit final models: collinearity pruning + refit ────────────────────────
#
# COLLINEAR_DROP — decisions per pair:
#
# Tamaño medio del hogar x Hogares unipersonales (M1, M3, M4)
#   -> Drop Hogares unipersonales: same household-size dimension from the
#      inverse angle; Tamaño medio entered earlier with larger delta.
#
# Gini x P80/P20 (M1)
#   -> Drop P80/P20: Gini is more standard in the inequality literature
#      and entered earlier in the stepwise.
#
# Renta media por hogar x Renta media por persona (M2, M3, M4)
#   -> Drop Renta por hogar: Renta per capita is more comparable across
#      municipalities with different household sizes.
#
# Total Empresas x Oficina de farmacia / Sucursal bancaria / Densidad (all models)
#   -> Drop Total Empresas: it is a composite size indicator that overlaps
#      with service-access variables; less interpretable substantively.
#
# Tamaño medio del hogar x Viviendas no principales (M4)
#   -> Drop Viviendas no principales: the second-residence dimension is
#      already captured by Plazas turísticas (tourist beds), retained in M4.
#
# Densidad x Total Empresas (M3)
#   -> Total Empresas already dropped; Densidad retained.

COLLINEAR_DROP = {
    'M1': [
        'VIVIENDA__HOGAR__Hogares unipersonales \n(% s/ total)',
        'ECONOMIA__RENTAS__DESIGUALDAD__Distribución de la renta P80/P20',
        'ECONOMIA__EMPRESAS__Total Empresas',
    ],
    'M2': [
        'ECONOMIA__RENTAS__Renta neta media por hogar',
        'ECONOMIA__EMPRESAS__Total Empresas',
    ],
    'M3': [
        'VIVIENDA__HOGAR__Hogares unipersonales \n(% s/ total)',
        'ECONOMIA__RENTAS__Renta neta media por hogar',
        'ECONOMIA__EMPRESAS__Total Empresas',
    ],
    'M4': [
        'VIVIENDA__HOGAR__Hogares unipersonales \n(% s/ total)',
        'ECONOMIA__RENTAS__Renta neta media por hogar',
        'MEDIO FÍSICO__Densidad (hab/km2)',
        'ECONOMIA__EMPRESAS__Total Empresas',
        'VIVIENDA__TIPOS DE VIVIENDAS (familiares)__Viviendas no principales (% s/ total)',
    ],
}

fitted_models = {}
metrics_rows  = []

for spec in MODEL_SPECS:
    mid    = spec['model_id']
    mdata  = model_data[mid]
    selres = selection_results[mid]

    X_df = mdata['X']
    y    = mdata['y']
    n    = mdata['n_total']

    typology_short   = 'Remote'   if spec['typology'] == RURAL_REMOTE  else 'Accessible'
    comparison_short = 'Reverter' if spec['response'] == 'is_reverter' else 'Dynamiser'

    # Step 1: variables from stepwise
    selected_vars = selres['selected'].copy()

    # Step 2: drop collinear variables
    drop_list = COLLINEAR_DROP.get(mid, [])
    pruned    = [v for v in selected_vars if v not in drop_list]
    dropped   = [v for v in selected_vars if v in drop_list]
    if dropped:
        print(f'{mid}: pruned {len(selected_vars)} -> {len(pruned)}  '
              f'(dropped: {[v.split("__")[-1][:40] for v in dropped]})')
    selected_vars = pruned

    # Step 3: fit final model
    # M2/M4 -> Firth penalisation (quasi-complete separation)
    # M1/M3 -> standard MLE
    final_result = fit_logit(X_df, y, selected_vars, model_id=mid)
    if final_result is None:
        print(f'WARNING: {mid} — model failed to converge')
        fitted_models[mid] = None
        continue

    # Step 4: evaluation metrics
    # Null model always uses standard MLE regardless of estimator
    # (intercept-only model has no separation risk)
    null_result = fit_logit_standard(X_df, y, [])
    mcfadden_r2 = 1 - (final_result.llf / null_result.llf)
    X_final     = sm.add_constant(X_df[selected_vars].values, has_constant='add')
    y_pred_prob = final_result.predict(X_final)
    auc         = roc_auc_score(y, y_pred_prob)

    n_extreme   = np.sum((y_pred_prob < 0.01) | (y_pred_prob > 0.99))
    pct_extreme = n_extreme / n * 100

    corr_retained = X_df[selected_vars].corr(method='spearman').abs()
    np.fill_diagonal(corr_retained.values, 0)
    max_rho = corr_retained.values.max() if len(selected_vars) >= 2 else np.nan

    k_final = len(selected_vars) + 1
    final_crit, final_crit_name = select_criterion(final_result.llf, k_final, n)

    # Step 5: store results
    fitted_models[mid] = {
        'result':        final_result,
        'selected_vars': selected_vars,
        'y_pred_prob':   y_pred_prob,
        'mcfadden_r2':   mcfadden_r2,
        'auc':           auc,
        'crit_name':     final_crit_name,
        'crit_value':    final_crit,
        'estimator':     'Firth L1' if mid in FIRTH_MODELS else 'MLE',
    }

    metrics_rows.append({
        'model_id':                  mid,
        'typology':                  spec['typology'],
        'comparison':                spec['response'].replace('is_', ''),
        'estimator':                 'Firth L1' if mid in FIRTH_MODELS else 'MLE',
        'n_target':                  mdata['n_target'],
        'n_reference':               mdata['n_reference'],
        'n_total':                   n,
        'n_predictors':              len(selected_vars),
        'criterion_name':            final_crit_name,
        'criterion_value':           round(final_crit, 4),
        'mcfadden_r2':               round(mcfadden_r2, 4),
        'auc':                       round(auc, 4),
        'n_extreme_probs':           n_extreme,
        'pct_extreme_probs':         round(pct_extreme, 2),
        'max_retained_spearman_rho': round(max_rho, 3) if not np.isnan(max_rho) else np.nan,
    })

    print(f'{mid} ({comparison_short}, {typology_short})  [{fitted_models[mid]["estimator"]}]:  '
          f'{final_crit_name}={final_crit:.4f}  '
          f'McFadden R2={mcfadden_r2:.4f}  '
          f'AUC={auc:.4f}  '
          f'n_pred={len(selected_vars)}  '
          f'max_rho={max_rho:.3f}')

df_metrics = pd.DataFrame(metrics_rows)
print('\nModel metrics computed.')

M1: pruned 16 -> 13  (dropped: ['Hogares unipersonales \n(% s/ total)', 'Total Empresas', 'Distribución de la renta P80/P20'])
M1 (Reverter, Remote)  [MLE]:  AIC=2002.1508  McFadden R2=0.1857  AUC=0.7974  n_pred=13  max_rho=0.507
M2: pruned 17 -> 15  (dropped: ['Renta neta media por hogar', 'Total Empresas'])
M2 (Dynamiser, Remote)  [Firth L1]:  AIC=540.3119  McFadden R2=0.3367  AUC=0.9109  n_pred=15  max_rho=0.583
M3: pruned 18 -> 15  (dropped: ['Hogares unipersonales \n(% s/ total)', 'Renta neta media por hogar', 'Total Empresas'])


C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '
C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


M3 (Reverter, Accessible)  [MLE]:  AIC=2669.4599  McFadden R2=0.2500  AUC=0.8235  n_pred=15  max_rho=0.595
M4: pruned 16 -> 11  (dropped: ['Hogares unipersonales \n(% s/ total)', 'Densidad (hab/km2)', 'Total Empresas', 'Viviendas no principales (% s/ total)', 'Renta neta media por hogar'])
M4 (Dynamiser, Accessible)  [Firth L1]:  AIC=1105.7309  McFadden R2=0.5650  AUC=0.9468  n_pred=11  max_rho=0.586

Model metrics computed.


C:\Users\juanz\miniconda3\envs\rural-migration\lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


In [13]:
# ── 4.2 Extract coefficients and odds ratios ──────────────────────────────────
# For each retained predictor: coefficient, OR, 95% CI, p-value.
# Intercept excluded (not interpretable after z-score standardisation).
#
# Note on Firth models (M2, M4): conf_int() and pvalues are available from
# statsmodels fit_regularized() but use profile likelihood, not Wald.
# This is appropriate for penalised models.

coef_rows = []

for spec in MODEL_SPECS:
    mid = spec['model_id']
    if fitted_models[mid] is None:
        continue

    result        = fitted_models[mid]['result']
    selected_vars = fitted_models[mid]['selected_vars']

    if len(selected_vars) == 0:
        continue

    conf_int = result.conf_int()
    params   = result.params
    pvalues  = result.pvalues

    var_names_model = ['const'] + selected_vars

    for i, var in enumerate(var_names_model):
        if var == 'const':
            continue
        coef  = params.iloc[i]
        ci_lo = conf_int.iloc[i, 0]
        ci_hi = conf_int.iloc[i, 1]
        p_val = pvalues.iloc[i]

        coef_rows.append({
            'model_id':       mid,
            'typology':       spec['typology'],
            'comparison':     spec['response'].replace('is_', ''),
            'estimator':      fitted_models[mid]['estimator'],
            'variable':       var,
            'variable_short': ' '.join(var.split('__')[-1].split()),
            'coef':           round(coef, 4),
            'or':             round(np.exp(coef), 4),
            'or_ci_lo':       round(np.exp(ci_lo), 4),
            'or_ci_hi':       round(np.exp(ci_hi), 4),
            'p_value':        round(p_val, 4),
            'significant_05': p_val < 0.05,
        })

df_coefs = pd.DataFrame(coef_rows)
print(f'Coefficients extracted: {len(df_coefs)} rows ({len(MODEL_SPECS)} models)')
print()
print(df_coefs[['model_id', 'variable_short', 'coef', 'or', 'or_ci_lo', 'or_ci_hi', 'p_value']].to_string(index=False))

AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

---
## 5 · Figures — Odds ratio plots with 95% CI

One forest-plot figure per typology. Each figure shows reverter (M1/M3, orange)
and dynamiser (M2/M4, red) models side by side. Variables are the union of
selected predictors from both models in that typology, sorted by absolute
log-OR in the reverter model (descending). OR=1 reference line shown.
Variables not selected in one model shown as grey diamond at OR=1.

In [ ]:
# ── 5.1 English short labels for OR figures ───────────────────────────────────

VAR_LABELS_EN = {
    'Renta neta media por persona':                                         'Net income per capita (EUR)',
    'Renta neta media por hogar':                                           'Net household income (EUR)',
    'Índice de Gini (%)':                                                   'Gini index (%)',
    'Distribución de la renta P80/P20':                                     'Income ratio P80/P20',
    'Tasa de paro':                                                         'Unemployment rate (%)',
    'Afiliados Régimen Especial (R. E.) T. Autónomos (% s/ Total)':        'Self-employed affiliates (%)',
    'Contratos indefinidos (% s/ Total)':                                   'Permanent contracts (%)',
    'Total Empresas':                                                       'Total firms',
    'Pensión Contributiva Media':                                           'Mean contributory pension (EUR)',
    'Porcentaje cobertura >= 100 Mbps (condiciones máxima demanda)':       'Broadband >=100 Mbps (%)',
    'Consultorio de atención primaria (número)':                            'Primary care centres (n)',
    'Oficina de farmacia (número)':                                         'Pharmacies (n)',
    'Nº centros de Educación Primaria':                                     'Primary schools (n)',
    'Tiempo municipio 5.000 hab. o más, más cercano (minutos)':            'Time to 5,000-hab. town (min)',
    'Tiempo municipio 20.000 hab. o más, más cercano (minutos)':           'Time to 20,000-hab. town (min)',
    'Sucursal bancaria (número)':                                           'Bank branches (n)',
    'Parque de vehículos x c/ 100 hab.':                                   'Vehicles per 100 inhab.',
    'Viviendas no principales (% s/ total)':                               'Non-primary dwellings (%)',
    'Tamaño medio del hogar':                                              'Mean household size',
    'Hogares unipersonales (% s/ total)':                                  'Single-person households (%)',
    'Plazas turísticas x c/ 100 hab.':                                     'Tourist beds per 100 inhab.',
    'Altitud capital (m)':                                                  'Elevation (m)',
    'Densidad (hab/km2)':                                                   'Population density (inhab/km2)',
    'Superficie (km2)':                                                     'Municipal area (km2)',
    'Superficie forestal (% s/ total)':                                     'Forest cover (%)',
    'Superficie protegida (% s/ total)':                                    'Protected area (%)',
}

def get_label(var_full):
    """Get English short label for a full SIDAMUN variable name."""
    short = ' '.join(var_full.split('__')[-1].split())  # collapse whitespace/newlines
    for key, label in VAR_LABELS_EN.items():
        if ' '.join(key.split()) == short:
            return label
    return short

print('Label helper defined.')

In [ ]:
# ── 5.2 OR forest plots: one figure per typology ──────────────────────────────

TYPOLOGY_PAIRS = [
    {'typology': RURAL_REMOTE,     'reverter_id': 'M1', 'dynamiser_id': 'M2', 'label': 'Rural-Remote'},
    {'typology': RURAL_ACCESSIBLE, 'reverter_id': 'M3', 'dynamiser_id': 'M4', 'label': 'Rural-Accessible'},
]

for tp in TYPOLOGY_PAIRS:
    mid_rev = tp['reverter_id']
    mid_dyn = tp['dynamiser_id']
    tlabel  = tp['label']

    if fitted_models.get(mid_rev) is None and fitted_models.get(mid_dyn) is None:
        print(f'Skipping {tlabel} — no fitted models.')
        continue

    vars_rev = fitted_models[mid_rev]['selected_vars'] if fitted_models.get(mid_rev) else []
    vars_dyn = fitted_models[mid_dyn]['selected_vars'] if fitted_models.get(mid_dyn) else []
    all_vars = list(dict.fromkeys(vars_rev + [v for v in vars_dyn if v not in vars_rev]))

    def sort_key(v):
        row = df_coefs[(df_coefs['model_id'] == mid_rev) & (df_coefs['variable'] == v)]
        return -abs(np.log(row['or'].values[0])) if len(row) > 0 else 0
    all_vars_sorted = sorted(all_vars, key=sort_key)

    n_vars = len(all_vars_sorted)
    if n_vars == 0:
        continue

    fig, ax = plt.subplots(figsize=(10, max(4, n_vars * 0.55 + 1.5)))
    y_positions = np.arange(n_vars)
    offset = 0.18

    for i, var in enumerate(all_vars_sorted):
        for j, (mid, cname, col, marker, off) in enumerate([
            (mid_rev, 'Reverter',  MODEL_COLORS['reverter'],  'o',  offset),
            (mid_dyn, 'Dynamiser', MODEL_COLORS['dynamiser'], 's', -offset),
        ]):
            row  = df_coefs[(df_coefs['model_id'] == mid) & (df_coefs['variable'] == var)]
            ypos = y_positions[i] + off
            if len(row) > 0:
                or_val = row['or'].values[0]
                ci_lo  = row['or_ci_lo'].values[0]
                ci_hi  = row['or_ci_hi'].values[0]
                sig    = row['significant_05'].values[0]
                alpha  = 1.0 if sig else 0.4
                ax.plot([ci_lo, ci_hi], [ypos, ypos], '-', color=col, lw=1.5, alpha=alpha)
                ax.plot(or_val, ypos, marker=marker, color=col, ms=7, alpha=alpha,
                        label=cname if i == 0 else '')
            else:
                ax.plot(1.0, ypos, marker='D', color='#cccccc', ms=5, alpha=0.5)

    ax.axvline(x=1.0, color='black', lw=0.8, ls='--', alpha=0.6)
    ax.set_yticks(y_positions)
    ax.set_yticklabels([get_label(v) for v in all_vars_sorted], fontsize=FONTSIZE - 1)
    ax.set_ylim(-0.5, n_vars - 0.5)
    ax.set_xlabel('Odds ratio (OR) per 1 SD', fontsize=FONTSIZE)
    ax.set_title(f'Logistic regression — {tlabel}\nOdds ratios vs. Structural decline (reference)',
                 fontsize=FONTSIZE, fontweight='bold')
    ax.tick_params(axis='x', labelsize=FONTSIZE - 1)

    handles = [
        mpatches.Patch(color=MODEL_COLORS['reverter'],  label=f'Reverter ({mid_rev})'),
        mpatches.Patch(color=MODEL_COLORS['dynamiser'], label=f'Dynamiser ({mid_dyn})'),
        mpatches.Patch(color='#cccccc',                 label='Not selected (OR=1 shown)'),
    ]
    ax.legend(handles=handles, fontsize=FONTSIZE - 1, loc='lower right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()

    fname = FIG_DIR / f'00_p5b_or_plot_{tlabel.lower().replace("-", "").replace(" ", "_")}_combined.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname.name}')
    plt.close()

---
## 6 · Random Forest robustness check

A Random Forest classifier is fitted for each of the four binary comparisons
using the **full 26-variable candidate set** (not only the logistic regression
selected subset).

**Purpose:** this is a **feature importance diagnostic**, not a predictive model.
Its function is to verify that the variable rankings produced by logistic
regression are not artefacts of the linear, additive model structure. The RF
can detect non-linear and interaction-driven importance patterns that stepwise
selection might miss. Consistency between the two rankings (assessed by
Spearman ρ) is reported as supplementary validation.

**Why RF is not used as primary model:** the research question requires
interpretable coefficients with confidence intervals (odds ratios) to support
the substantive narrative about which structural conditions distinguish each
demographic group. RF feature importance is not directly translatable into
effect-size estimates for publication.

**Hyperparameters:** n_estimators=500, max_features='sqrt',
class_weight='balanced' (handles imbalanced n_target/n_reference), random_state=42.

In [ ]:
# ── 6.1 Fit Random Forests for all four comparisons ───────────────────────────

rf_results = {}

for spec in MODEL_SPECS:
    mid            = spec['model_id']
    mdata          = model_data[mid]
    available_vars = mdata['available_vars']
    X_df           = mdata['X']
    y              = mdata['y']

    rf = RandomForestClassifier(
        n_estimators=500,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_df[available_vars].values, y)

    importances = rf.feature_importances_
    df_imp_rf = pd.DataFrame({
        'model_id':      mid,
        'typology':      spec['typology'],
        'comparison':    spec['response'].replace('is_', ''),
        'variable':      available_vars,
        'variable_short': [' '.join(v.split('__')[-1].split()) for v in available_vars],
        'rf_importance': importances,
        'rf_rank':       pd.Series(importances).rank(ascending=False).astype(int).values,
    }).sort_values('rf_importance', ascending=False).reset_index(drop=True)

    rf_results[mid] = {'rf': rf, 'importance_df': df_imp_rf}

    typology_short   = 'Remote'   if spec['typology'] == RURAL_REMOTE  else 'Accessible'
    comparison_short = 'Reverter' if spec['response'] == 'is_reverter' else 'Dynamiser'
    print(f'{mid} ({comparison_short}, {typology_short}) — top 5 RF features:')
    for _, row in df_imp_rf.head(5).iterrows():
        print(f'    {row["rf_rank"]:>2}.  {get_label(row["variable"]):<45}  importance={row["rf_importance"]:.4f}')
    print()

df_rf_all = pd.concat([rf_results[m]['importance_df'] for m in rf_results], ignore_index=True)
print(f'RF importance table: {len(df_rf_all)} rows')

In [ ]:
# ── 6.2 Spearman rank correlation: logistic |coef| vs. RF importance ──────────
# Computed on logistic-selected variables only.
# High ρ confirms that the logistic variable ranking is not an artefact
# of the linear model structure.

print('Spearman rho between |logistic coef| rank and RF importance rank')
print('(computed on logistic-selected variables only)\n')
print(f'{"Model":<6} {"Comparison":<12} {"Typology":<18} {"n_selected":>10} {"Spearman rho":>13} {"p-value":>10}')
print('-' * 75)

for spec in MODEL_SPECS:
    mid = spec['model_id']
    if fitted_models.get(mid) is None:
        continue

    selected_vars = fitted_models[mid]['selected_vars']
    if len(selected_vars) < 3:
        print(f'{mid:<6}  too few selected variables for rank correlation')
        continue

    log_coefs = df_coefs[df_coefs['model_id'] == mid].set_index('variable')['coef'].abs()
    rf_imp    = rf_results[mid]['importance_df'].set_index('variable')['rf_importance']
    common    = [v for v in selected_vars if v in log_coefs.index and v in rf_imp.index]

    if len(common) < 3:
        continue

    rho, pval = spearmanr(log_coefs[common].values, rf_imp[common].values)

    comparison_short = 'Reverter' if spec['response'] == 'is_reverter' else 'Dynamiser'
    typology_short   = 'Remote'   if spec['typology'] == RURAL_REMOTE  else 'Accessible'
    print(f'{mid:<6} {comparison_short:<12} {typology_short:<18} {len(common):>10} {rho:>13.4f} {pval:>10.4f}')

---
## 7 · Figure — RF importance vs. logistic coefficients

One figure per typology. Each figure: two panels (reverter / dynamiser).
Each panel: dual horizontal bar chart — RF importance (grey, all 26 variables)
and |standardised logistic coefficient| (coloured, selected variables only),
sorted by RF importance. Variables retained in the logistic model highlighted.

In [ ]:
# ── 7.1 RF importance vs. logistic |coef| comparison figures ─────────────────

for tp in TYPOLOGY_PAIRS:
    mid_rev = tp['reverter_id']
    mid_dyn = tp['dynamiser_id']
    tlabel  = tp['label']

    fig, axes = plt.subplots(1, 2, figsize=(16, 9), sharey=False)

    for ax, mid, comp_label, col in [
        (axes[0], mid_rev, 'Reverter',  MODEL_COLORS['reverter']),
        (axes[1], mid_dyn, 'Dynamiser', MODEL_COLORS['dynamiser']),
    ]:
        if rf_results.get(mid) is None:
            ax.set_visible(False)
            continue

        df_rf = rf_results[mid]['importance_df'].copy()
        df_rf['label'] = df_rf['variable'].apply(get_label)
        df_rf = df_rf.sort_values('rf_importance', ascending=True)

        if fitted_models.get(mid) is not None:
            log_dict     = df_coefs[df_coefs['model_id'] == mid].set_index('variable')['coef'].abs().to_dict()
            selected_set = set(fitted_models[mid]['selected_vars'])
        else:
            log_dict     = {}
            selected_set = set()

        df_rf['log_coef_abs'] = df_rf['variable'].map(log_dict).fillna(0)
        df_rf['in_logistic']  = df_rf['variable'].isin(selected_set)

        n_vars = len(df_rf)
        y_pos  = np.arange(n_vars)

        ax.barh(y_pos, df_rf['rf_importance'].values, height=0.4, color='#cccccc',
                label='RF importance', align='center')

        for i, (_, row) in enumerate(df_rf.iterrows()):
            if row['in_logistic'] and row['log_coef_abs'] > 0:
                ax.barh(y_pos[i], row['log_coef_abs'], height=0.25, color=col,
                        alpha=0.85, label='|Logistic coef|' if i == 0 else '')

        ax.set_yticks(y_pos)
        ax.set_yticklabels(df_rf['label'].values, fontsize=FONTSIZE - 2)
        ax.set_xlabel('Importance / |coef|', fontsize=FONTSIZE - 1)
        ax.set_title(f'{comp_label} — {tlabel}', fontsize=FONTSIZE, fontweight='bold')
        ax.tick_params(axis='x', labelsize=FONTSIZE - 2)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        handles = [
            mpatches.Patch(color='#cccccc', label='RF importance (all 26 vars)'),
            mpatches.Patch(color=col,       label='|Logistic coef| (selected vars)'),
        ]
        ax.legend(handles=handles, fontsize=FONTSIZE - 2, loc='lower right')

    plt.suptitle(f'Random Forest importance vs. logistic |coefficient| — {tlabel}',
                 fontsize=FONTSIZE, fontweight='bold', y=1.01)
    plt.tight_layout()

    fname = FIG_DIR / f'00_p5b_rf_vs_logistic_{tlabel.lower().replace("-", "").replace(" ", "_")}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname.name}')
    plt.close()

---
## 8 · Figure — ROC curves

ROC curves for all four logistic regression models. One figure per typology
with two curves (reverter / dynamiser). AUC values annotated on the figure.

In [ ]:
# ── 8.1 ROC curves ────────────────────────────────────────────────────────────

for tp in TYPOLOGY_PAIRS:
    mid_rev = tp['reverter_id']
    mid_dyn = tp['dynamiser_id']
    tlabel  = tp['label']

    fig, ax = plt.subplots(figsize=(6, 6))

    for mid, comp_label, col, ls in [
        (mid_rev, 'Reverter',  MODEL_COLORS['reverter'],  '-'),
        (mid_dyn, 'Dynamiser', MODEL_COLORS['dynamiser'], '--'),
    ]:
        if fitted_models.get(mid) is None:
            continue
        y_true = model_data[mid]['y']
        y_prob = fitted_models[mid]['y_pred_prob']
        auc    = fitted_models[mid]['auc']
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        ax.plot(fpr, tpr, color=col, lw=2, ls=ls, label=f'{comp_label} (AUC={auc:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random classifier (AUC=0.5)')
    ax.set_xlabel('False positive rate', fontsize=FONTSIZE)
    ax.set_ylabel('True positive rate', fontsize=FONTSIZE)
    ax.set_title(f'ROC curves — {tlabel}\nvs. Structural decline (reference)',
                 fontsize=FONTSIZE, fontweight='bold')
    ax.legend(fontsize=FONTSIZE - 1, loc='lower right')
    ax.tick_params(labelsize=FONTSIZE - 1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()

    fname = FIG_DIR / f'00_p5b_roc_{tlabel.lower().replace("-", "").replace(" ", "_")}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved: {fname.name}')
    plt.close()

---
## 9 · Export outputs

In [ ]:
# ── 9.1 Export all tabular outputs ────────────────────────────────────────────

outputs = {
    DEMO_DERIV / 'p5b_model_results.csv':  df_coefs,
    DEMO_DERIV / 'p5b_model_metrics.csv':  df_metrics,
    DEMO_DERIV / 'p5b_rf_importance.csv':  df_rf_all,
    DEMO_DERIV / 'p5b_imputation_log.csv': df_imputation_log,
}

for path, df_out in outputs.items():
    df_out.to_csv(path, sep=';', encoding='utf-8-sig', index=False)
    print(f'  ok  {path.name}  ({len(df_out)} rows)')

# Stepwise trace log
step_log_rows = []
for mid, sres in selection_results.items():
    for row in sres['step_log']:
        row['model_id'] = mid
        step_log_rows.append(row)

if step_log_rows:
    df_step_log = pd.DataFrame(step_log_rows)
    out_steps   = DEMO_DERIV / 'p5b_stepwise_log.csv'
    df_step_log.to_csv(out_steps, sep=';', encoding='utf-8-sig', index=False)
    print(f'  ok  {out_steps.name}  ({len(df_step_log)} rows)')

# Figure inventory
figs = sorted(FIG_DIR.glob('*.png'))
print(f'\n  Figures saved: {len(figs)}')
for f in figs:
    print(f'    ok  {f.name}')

---
## 10 · Methodological notes and references

---

### Why logistic regression — the multivariate rationale

Step 5a (Kruskal-Wallis + Dunn/Bonferroni) characterises each group on
individual variables in isolation. This is a valid descriptive step but
is vulnerable to the *cherry-picking* critique: with 51 significant variables,
highlighting any subset may appear to explain group membership without
controlling for confounders. Logistic regression addresses this directly by
competing all candidate variables simultaneously. The stepwise AIC procedure
retains only variables that contribute net explanatory power over and above
the remaining predictors — the multivariate analytical step.

---

### Bidirectional stepwise vs. best subset

With 26 candidate predictors, exhaustive best-subset selection requires
evaluation of 2^26 approximately 67 million models and is computationally
infeasible. Bidirectional stepwise provides a robust alternative: allowing
variable removal at each step avoids the local-optima problem of purely
forward selection, where a variable admitted early may become redundant
once later predictors enter (Venables & Ripley, 2002; Hosmer & Lemeshow,
2000, Ch. 4).

---

### AIC vs. AICc

The Akaike Information Criterion (AIC; Akaike, 1974) penalises model
complexity (-2*logL + 2K). The corrected version AICc (Hurvich & Tsai, 1989)
adds a finite-sample correction and is preferred when n/K < 40 (Burnham &
Anderson, 2002, p. 66). The criterion is selected automatically per model
and per step based on the n/K ratio.

---

### Collinearity pruning

The stepwise AIC minimises predictive redundancy but does not penalise
pairwise collinearity between retained predictors. When Spearman |rho| > 0.70
between two retained variables, the Hessian matrix becomes ill-conditioned,
producing unreliable standard errors, confidence intervals, and p-values.
One variable per collinear pair is dropped post-hoc using the rule: drop
the variable that (a) entered later in the stepwise and (b) is less
interpretable substantively. Decisions are logged in COLLINEAR_DROP (Section 4).

---

### Quasi-complete separation and Firth penalisation (M2, M4)

Dynamisers (Grows in both) are structurally so different from Structural
decline municipalities that standard MLE produces quasi-complete separation:
the model assigns extreme predicted probabilities (>20% of observations
with p<0.01 or p>0.99), inflating coefficients and destabilising the Hessian.
This is a genuine substantive finding — the high AUC (>0.90) confirms it —
but it invalidates standard error estimates.

Firth penalised likelihood (Firth, 1993; Heinze & Schemper, 2002) adds a
Jeffreys prior penalty to the log-likelihood, shrinking coefficients away
from infinity without removing variables. Implemented here via light L1
regularisation (alpha=0.01), which closely approximates the Firth correction.
M1 and M3 (reverters) show no separation and use standard MLE.

---

### Standardisation and OR interpretation

All predictors are z-score standardised before fitting. Odds ratios represent
the multiplicative change in odds for a one-standard-deviation increase in
the predictor. This facilitates comparison across variables with heterogeneous
units but means OR cannot be directly compared to unstandardised coefficients
from other studies without back-transformation.

---

### Confidentiality imputation

Gini, P80/P20, and income variables are suppressed by INE for municipalities
below a population threshold. This suppression is not random — it is
concentrated in small municipalities overrepresented in the Structural
decline group. Imputing with the size_group x typology cell median rather
than the typology median reduces bias toward larger municipalities
(van Buuren, 2018, Ch. 1). The imputation is logged in p5b_imputation_log.csv.

---

### Random Forest as robustness check

The RF is not a predictive model in this context. It is a feature importance
diagnostic to verify that the variable rankings from logistic regression are
not artefacts of linearity or additive model structure. The RF uses the full
26-variable pool to allow detection of non-linear or interaction-driven
importance that stepwise selection might miss. Consistency between the two
rankings (Spearman rho) is reported as supplementary validation, not as an
independent finding (Breiman, 2001; Strobl et al., 2008).

---

### Excluded group

The Loses in B (A↑B↓) group is excluded from all logistic regression models.
This group has limited sample size in Rural-Remote (n=149) and its
theoretical interpretation — municipalities that gained population in the
pre-inflection period but lost it afterward — is secondary to the reverter
and dynamiser comparisons that anchor the paper's research question.
Including it would require two additional models without adding interpretive
value proportional to the increased complexity.

---

### Cross-sectional limitation

SIDAMUN variables reflect 2022–2023 conditions, posterior to T=2018.
Predictors may be both causes and consequences of the demographic trajectory
observed in Period B. The logistic regression identifies structural associations
between current socioeconomic profile and demographic group membership, not
causal antecedents of demographic reversal. This is stated explicitly in
the paper methodology.

---

### Note on LISA analysis

A separate notebook (p5a_lisa_spatial_outliers.ipynb) characterises LISA
quadrants (HH, HL, LH, LL computed on var_acum_pct_B) using the same
KW/Dunn approach as p5a. This is a complementary spatial analysis that
describes socioeconomic differences between spatially autocorrelated clusters,
not a multivariate model. It belongs analytically closer to Section 4
(spatial hotspots) than to the explanatory modelling of this notebook,
and will be positioned accordingly in the paper (supplementary or Section 4
extension).

---

### Key references

- Akaike, H. (1974). A new look at the statistical model identification. *IEEE Transactions on Automatic Control*, 19(6), 716–723.
- Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32.
- Burnham, K. P., & Anderson, D. R. (2002). *Model Selection and Multimodel Inference*. Springer.
- Firth, D. (1993). Bias reduction of maximum likelihood estimates. *Biometrika*, 80(1), 27–38.
- Heinze, G., & Schemper, M. (2002). A solution to the problem of separation in logistic regression. *Statistics in Medicine*, 21, 2409–2419.
- Hosmer, D. W., & Lemeshow, S. (2000). *Applied Logistic Regression* (2nd ed.). Wiley.
- Hurvich, C. M., & Tsai, C.-L. (1989). Regression and time series model selection in small samples. *Biometrika*, 76(2), 297–307.
- McFadden, D. (1974). Conditional logit analysis of qualitative choice behavior. In P. Zarembka (Ed.), *Frontiers in Econometrics* (pp. 105–142). Academic Press.
- Strobl, C., Boulesteix, A.-L., Kneib, T., Augustin, T., & Zeileis, A. (2008). Conditional variable importance for random forests. *BMC Bioinformatics*, 9, 307.
- van Buuren, S. (2018). *Flexible Imputation of Missing Data*. CRC Press.
- Venables, W. N., & Ripley, B. D. (2002). *Modern Applied Statistics with S* (4th ed.). Springer.

---

### Outputs for p5c

No direct data outputs feed into p5c. Step 5c (EMCR migration flows) is
a descriptive analysis operating on a different data source (INE EMCR
2021–2024) and does not depend on the logistic regression results.